In [ ]:
from sh import esearch, efetch
from io import StringIO
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from fennec.featurizer import helm_one_hot_encode, helm_one_hot_decode
from Bio.Seq import reverse_complement_rna

fasta_id = 'NM_012111.3'
fasta_path = f'{fasta_id}.fna'

#fasta_buffer = open(fasta_path, 'w')
#efetch('-db', 'nuccore', '-id', fasta_id, '-format', 'fasta', _out=fasta_buffer)

In [ ]:
from Bio import SeqIO
from Bio.Seq import reverse_complement_rna
fasta_path = f'{fasta_id}.fna'
record = SeqIO.read(fasta_path, 'fasta')
record

In [ ]:
design_test = pd.read_csv('Ahsa1_test_designs.csv')

In [ ]:
chem_patterns = pd.read_csv('chemical_patterns_paddle.csv')
chem_pat_df_pass = pd.read_csv('chemical_patterns_passenger_paddle.csv')
guide_inds_nums = chem_patterns['0'].to_dict()
pass_inds_nums = chem_pat_df_pass['0'].to_dict()
chem_patterns.head(1)

In [ ]:
from glob import glob
model_paths = glob('weights/*_model_output.pt')

In [ ]:
from fennec.preprocessing import preprocess_input_tracks

no_passenger = False
with_chem_ablate = False
with_thermo = True
fm_type = 'rna_fm'
slice_length = 25
only_cls_eos=False
device='cuda'

rna_emb_path = '.'

X, y, X_meta, X_emb, mrna_map_index = preprocess_input_tracks(design_test, 
                                                    rna_emb_path, 
                                                    device=device,
                                                    with_passenger=not no_passenger,
                                                    with_chem_ablate=with_chem_ablate,
                                                    with_thermo=with_thermo,
                                                    fm_type=fm_type,
                                                    slice_length=slice_length,
                                                    only_cls_eos=only_cls_eos)
X_emb = X_emb[mrna_map_index].to(device)

In [ ]:
from fennec.models import MODELS

In [ ]:
import os
pt1 = model_paths[0]
os.path.basename(pt1)

In [ ]:
import torch
from tqdm import tqdm
from torch.utils.data import DataLoader, TensorDataset

BATCH_SIZE = 128
NUM_WORKERS = 1

model_preds = {}

for w_path in tqdm(model_paths):
    model_name = os.path.basename(w_path).strip('.pt')
    n_X, total_channels, seq_len = X.shape
    meta_input_dim = X_meta.shape[-1]
    nembs, fm_seq_len, fm_embed_dim = X_emb.shape
    model = MODELS['TCN_FM_multi_head'](input_channels=total_channels, 
                   meta_input_dim=meta_input_dim,
                    fm_seq_len=fm_seq_len,
                    fm_embed_dim=fm_embed_dim,
                    num_classes=1,
                   regression=True,
                   ).to(device)
    
    model.load_state_dict(torch.load(w_path, weights_only=False))
    model.eval()
    y_pred = model(X.to(device), X_meta.to(device), X_emb.to(device)).squeeze(1).to('cpu').detach().numpy()
    model_preds[model_name] = y_pred
    del model


In [ ]:
design_predictions = pd.DataFrame(model_preds)
design_predictions['stdev'] = design_predictions.std(axis=1)
design_predictions['mean_pred'] = design_predictions.mean(axis=1)

In [ ]:
sns.histplot(data=design_predictions, x= 'mean_pred')

In [ ]:
sns.histplot(data=design_predictions, x= 'stdev')

In [ ]:
design_pred_df = pd.concat([design_test, design_predictions], axis=1)
design_pred_df['guide_pat_ind'] = design_pred_df['guide_passenger_pair'].map(lambda x: x.split(',')[0].strip('()'))
design_pred_df['passenger_pat_ind'] = design_pred_df['guide_passenger_pair'].map(lambda x: x.split(',')[1].strip('()'))

mess_locs = []
mess_perc_loc = []
for si, mr in zip(design_pred_df.siRNA, design_pred_df.mRNA):
    loc= mr.replace('T', 'U').find(reverse_complement_rna(si))
    
    mess_locs.append(loc)
    mess_perc_loc.append(loc/len(mr))

design_pred_df['siRNA_location'] = mess_locs
design_pred_df['siRNA_percent_loc'] = mess_perc_loc

#design_pred_df = design_pred_df[design_pred_df.stdev < 7.3]
#sns.histplot(data=design_pred_df, x='mean_pred')

best_pair = design_pred_df.groupby('guide_passenger_pair')['mean_pred'].mean().sort_values(ascending=False).index[-1]
best_pair

In [ ]:
plt.figure(dpi=200, figsize=(15, 5))
design_pred_df.groupby('guide_passenger_pair')['mean_pred'].mean().sort_values(ascending=False).plot.bar()
plt.xticks(rotation=45, ha='right')
plt.ylabel('guide_passenger_pattern Mean % Knockdown Predicted')
plt.show()

plt.figure(dpi=200, figsize=(5, 5))
design_pred_df.groupby('guide_pat_ind')['mean_pred'].mean().sort_values(ascending=False).plot.bar()
plt.xticks(rotation=0, ha='center')
plt.ylabel('Guide Pattern Mean % Knockdown Predicted')
plt.show()

plt.figure(dpi=200, figsize=(7, 5))
design_pred_df.groupby('passenger_pat_ind')['mean_pred'].mean().sort_values(ascending=False).plot.bar()
plt.xticks(rotation=0, ha='center')
plt.ylabel('Passenger Pattern Mean % Knockdown Predicted')
plt.show()

plt.figure(dpi=200, figsize=(4, 7))
design_pred_df.groupby('guide_passenger_pair')['mean_pred'].count().sort_values(ascending=False).plot.barh()
plt.xlabel('N patterns designed')
pass

In [ ]:
ddf79 = design_pred_df[design_pred_df.guide_passenger_pair == best_pair]
sns.histplot(data=ddf79, x='mean_pred')

In [ ]:
sns.histplot(data=ddf79, x='stdev')

In [ ]:
ddf79 = ddf79[ddf79.stdev <= 12]
sns.histplot(data=ddf79, x='mean_pred')

In [ ]:
pd.options.display.max_colwidth = 100
ddf79.sort_values(by='mean_pred', inplace=True)
plt.figure(dpi=100, figsize=(10, 4))
ddf79.mean_pred.plot.bar()
ddf79

In [ ]:
sns.histplot(data= ddf79, x='siRNA_percent_loc', bins=100)
plt.xlabel('siRNA mRNA location')
plt.xlim(0, 1)

In [ ]:
ddf79['bin_perc_loc'] = pd.cut(ddf79['siRNA_percent_loc'], 5)


plt.figure(dpi=200)
sns.histplot(data= ddf79, x='mean_pred', hue='bin_perc_loc', bins=10, multiple='dodge')


In [ ]:
ddf79[['mean_pred', 'stdev', 'siRNA_percent_loc']].corr()

In [ ]:
sns.scatterplot(data=ddf79, x='mean_pred', y='stdev')

In [ ]:
ddf79.head(50).siRNA_percent_loc.plot.hist()

In [ ]:
from primer3.bindings import calc_tm
ddf79['TM'] = ddf79.siRNA.map(lambda x: calc_tm(x.replace('U', 'T')))
ddf79['mean_conf'] = ddf79.mean_pred + (1.96*ddf79.stdev)
best = ddf79.sort_values(by='mean_conf').head(60)
best['prediction_class'] = 'Strong knockdown'
best

In [ ]:
std_cutoff = 12
middle_worse = ddf79[ddf79.stdev < std_cutoff].sort_values(by='mean_pred').tail(25)
middle_worse['prediction_class'] = 'Weak knockdown'
ddf79[ddf79.stdev < std_cutoff].sort_values(by='mean_pred').tail(25).siRNA_percent_loc.plot.hist(bins=10)

plt.show()

ddf79[ddf79.stdev < std_cutoff].sort_values(by='mean_pred').tail(25).mean_pred.plot.bar()

In [ ]:
final_des = pd.concat([best, middle_worse]).drop(columns='mRNA')
final_des

In [ ]:
final_des.mean_pred.plot.hist(bins=20)

In [ ]:
final_des.to_csv(f'Ahsa1-{best_pair}_pattern_design_siRNA_N=85.csv')

In [ ]:
ddf79.to_csv(f'Ahsa-{best_pair}_pattern_design_full.csv')